# scale_grid

> Phase 14 grids and the analysis plans: `trustGrid` (Part A, the private record and the public check), `scaleGrid` (Part B, the team-size sweep with a quarter of the agents liars), `scalePilotGrid`, quick grids, `TRUST_PLAN`, `SCALE_PLAN`, call estimates and the pre-set cost rule.

In [ ]:
#| default_exp scale_grid

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import copy

from explRL.clue_grid import CLUE_SETTINGS
from explRL.fix_grid import FIX_ARMS, FIX_SEEDS, fixCell, fixEstimate, fixGrid, phase11Cells
from explRL.llm_budget import PRICES
from explRL.llm_grid import HAIKU

TRUST_ARMS = ('recordPrivate', 'verifyPublic')                    # new; record, verify and baseline: Phase 13's fixA rows
SCALE_N, SCALE_SEEDS = (8, 16, 32), range(900, 905)
SCALE_CLEAN, SCALE_LIAR = ('baseline', 'request'), ('baseline', 'record')
TRUST_PLAN = dict(
    version=1, frozen=False, model=HAIKU, settings=CLUE_SETTINGS, prices=PRICES, turns=40, N=4, channel='blackboard',
    arms=dict(new=list(TRUST_ARMS), reused=['baseline', 'record', 'verify'],
              world={a: FIX_ARMS[a] for a in ('baseline', 'record', 'verify', *TRUST_ARMS)},
              public=['record', 'verifyPublic'], private=['recordPrivate', 'verify'], free=['record', 'recordPrivate'],
              costly=['verify', 'verifyPublic']),
    liar=dict(nLiar=1, slot=0, agent='ClueLiar (scripted, Phase 11)'),
    seeds=dict(trust=[900, 909]), reuse="record, verify and baseline rows: Phase 13's study fixA (same cells), joined by seed",
    holm=list(TRUST_ARMS),
    analyses=dict(
        lossRecovered='recordPrivate and verifyPublic: loss recovered = 1 - mean honest loss (welfareHonest clean - liar, '
                      "same seed and arm) / baseline's, N 4, bootstrap CI over seeds; Holm over the 2 tests",
        publicity='(LR(record) + LR(verifyPublic)) / 2 - (LR(recordPrivate) + LR(verify)) / 2, each LR from the arms\' mean '
                  'honest losses; CI by bootstrap over seeds, the same seeds resampled for every arm (H14-1: above 0)',
        cost='(LR(record) + LR(recordPrivate)) / 2 - (LR(verify) + LR(verifyPublic)) / 2, likewise',
        difference='publicity effect minus cost effect, the same bootstrap (H14-2: above 0)',
        noLiarCost="each arm's no-liar welfare minus baseline's no-liar welfare, paired on seed",
        mechanism='checks and checks found false (turn logs); for recordPrivate, the mean checked count behind the '
                  "'record a/b true' entries agents saw (parsed from observations)"),
    nboot=2000, ci=0.95, exploratory=True)
SCALE_PLAN = dict(
    version=1, frozen=False, model=HAIKU, settings=CLUE_SETTINGS, prices=PRICES, turns=40, channel='blackboard',
    N=list(SCALE_N), liarShare=0.25, liar=dict(nLiar='N // 4', slots='0 .. N // 4 - 1', agent='ClueLiar (scripted, Phase 11)'),
    arms=dict(clean=list(SCALE_CLEAN), liar=list(SCALE_LIAR), world={a: FIX_ARMS[a] for a in ('baseline', 'request', 'record')}),
    seeds=dict(scale=[900, 904], pilot=[900, 900], cached='N 8 clean baseline and request, seeds 900-904: Phase 13 cells'),
    pilot=dict(N=32, arms=['baseline'], liar=[0, 8], seeds=[900]),
    bands=dict(invalid=0.05, N=32),
    costRule=dict(budget=40.0, cutN=32, cutSeeds=[900, 902], otherwise='stop and ask',
                  projection="the main grid's new calls x the pilot's measured cost per call at N 32 (an upper bound for "
                             'N 8 and 16); the pilot cells are cached'),
    holm=dict(request=list(SCALE_N), record=list(SCALE_N)),
    analyses=dict(
        request="per N: request's pooled-share gap closed vs baseline (clean, ceiling 1), paired on seed, bootstrap CI "
                'over seeds; Holm over the 3 N (H14-3: below 0 at every N)',
        record="per N: record's loss recovered = 1 - mean honest loss (welfareHonest clean - with N // 4 liars, same N, "
               "seed and arm) / baseline's, bootstrap CI over seeds; Holm over the 3 N (H14-4: above 0 at every N)",
        baseline="the baseline's pooled share (clean) and honest loss per N; per arm, the trend across N",
        forecast='tools/scale_forecast.py score: the numpy forecast of the baseline trends N 8 -> 32 against the Haiku '
                 'baseline cells (H14-5)'),
    nboot=2000, ci=0.95, exploratory=True)


def trustGrid(seeds=FIX_SEEDS, s=CLUE_SETTINGS):
    "Study trustA: recordPrivate and verifyPublic at N 4, with one liar (slot 0) and without, 40 turns."
    return [fixCell('trustA', a, 4, sd, liar, s) for a in TRUST_ARMS for liar in (False, True) for sd in seeds]


def scaleGrid(seeds=SCALE_SEEDS, seeds32=None, s=CLUE_SETTINGS, study='scaleB', sizes=SCALE_N):
    """Study scaleB: blackboard, N 8, 16, 32, 40 turns; clean baseline and request; baseline and record with N // 4
    liars. seeds32: the seeds at N 32 (the cost rule's cut), else `seeds`."""
    sd = lambda N: (seeds32 if seeds32 is not None and N == 32 else seeds)
    return [fixCell(study, a, N, x, liar and N // 4, s) for liar, arms in ((False, SCALE_CLEAN), (True, SCALE_LIAR))
            for a in arms for N in sizes for x in sd(N)]


def scalePilotGrid(s=CLUE_SETTINGS):
    "N 32, baseline, clean and with 8 liars, seed 900 (the main run replays these replies)."
    return [fixCell('scalePilot', 'baseline', 32, 900, liar, s) for liar in (False, 8)]


def _short(cells, turns):
    return [dict(c, cfg=dict(copy.deepcopy(c['cfg']), turns=turns)) for c in cells]


def trustQuick(turns=6):
    "trustGrid's cells at seeds 900-901, a few turns, for the fake-client pipeline check."
    return _short(trustGrid((900, 901)), turns)


def scaleQuick(turns=3):
    "scaleGrid's cells at seed 900, a few turns."
    return _short(scaleGrid((900,)), turns)


def scaleCached():
    "Cells whose requests earlier runs made (Phase 11, Phase 13's fixA, the pilot): free on a rerun."
    return phase11Cells() + fixGrid() + scalePilotGrid()


def scaleEstimate(cells):
    "fixEstimate with the Phase 11, Phase 13 and pilot cells as cached."
    return fixEstimate(cells, scaleCached())


def costRule(perCall, plan=SCALE_PLAN):
    """The pre-set cost rule (FR-B2): projected = new calls x perCall ($, the pilot's at N 32). Within the budget: run the
    grid; else N 32 on the cut seeds if that fits; else stop and ask. -> dict(verdict, projected, cells or None)."""
    r, lo, hi = plan['costRule'], *plan['seeds']['scale']
    full, cut = scaleGrid(range(lo, hi + 1)), scaleGrid(range(lo, hi + 1), range(r['cutSeeds'][0], r['cutSeeds'][1] + 1))
    proj = {k: scaleEstimate(g)['newCalls'] * perCall for k, g in (('full', full), ('cut', cut))}
    verdict = 'run' if proj['full'] <= r['budget'] else 'cut' if proj['cut'] <= r['budget'] else 'stop'
    return dict(verdict=verdict, projected=proj, cells=dict(run=full, cut=cut).get(verdict))

## Tests

In [ ]:
import json
from explRL.scale_grid import *
from explRL.bench_grid import rowKey
from explRL.clue_grid import BOARD, CLUE_SETTINGS
from explRL.coop_grid import coopHash
from explRL.fix_grid import FIX_ARMS, fixEstimate, fixGrid, requestKey
from explRL.llm_runs import LLMConfig, cellRow

key = lambda c: rowKey(cellRow(c, LLMConfig.fromDict(c['cfg']), dict(episode=0)))
lv = lambda c: c['level']
# Part A (AC-3): 2 new arms x {clean, one liar} x N 4 x seeds 900-909, 40 turns, Phase 13's board and settings
t = trustGrid()
assert len(t) == 2 * 2 * 10 and len({key(c) for c in t}) == len(t) and all(c['study'] == 'trustA' for c in t)
assert {lv(c) for c in t} == {'recordPrivate', 'recordPrivate|liar', 'verifyPublic', 'verifyPublic|liar'}
for c in t:
    cfg, w = c['cfg'], c['cfg']['world']
    assert (cfg['channel'], cfg['turns'], cfg['model'], c['N'], cfg['nLiar']) == ('blackboard', 40, HAIKU, 4, lv(c).endswith('|liar'))
    assert {k: w[k] for k in BOARD} == {k: CLUE_SETTINGS[k] for k in BOARD}
    assert {k: v for k, v in w.items() if k not in BOARD} == FIX_ARMS[lv(c).split('|')[0]]
for x in {lv(c) for c in t}:
    assert sorted(c['seed'] for c in t if lv(c) == x) == list(range(900, 910))     # shared seeds
# ... the same cells as Phase 13's A2 arms except the arm's setting (study, level and the option differ)
f13 = {(c['level'], c['seed']): c for c in fixGrid()}
for c in t:
    ref = f13[(lv(c).replace('recordPrivate', 'record').replace('verifyPublic', 'verify'), c['seed'])]
    strip = lambda c: {k: v for k, v in c['cfg']['world'].items() if k not in ('record', 'check')}
    assert {k: v for k, v in c['cfg'].items() if k != 'world'} == {k: v for k, v in ref['cfg'].items() if k != 'world'}
    assert strip(c) == strip(ref) and requestKey(c) != requestKey(ref)
e = scaleEstimate(t)
assert e['calls'] == e['newCalls'] == 20 * 160 + 20 * 120 == 5600 and 1.2 < e['newCost'] < 2.2, e   # about $1.70

# Part B (AC-3): N 8, 16, 32; clean baseline and request; baseline and record with N // 4 liars; seeds 900-904
g = scaleGrid()
assert len(g) == 4 * 3 * 5 and len({key(c) for c in g}) == len(g) and all(c['study'] == 'scaleB' for c in g)
assert {c['N'] for c in g} == {8, 16, 32} and {c['seed'] for c in g} == set(range(900, 905))
assert all(c['cfg']['nLiar'] == (c['N'] // 4 if lv(c).endswith('|liar') else 0) for c in g)
assert {c['cfg']['nLiar'] for c in g if lv(c).endswith('|liar')} == {2, 4, 8}
assert {lv(c) for c in g} == {'baseline', 'request', 'baseline|liar', 'record|liar'}
assert all(c['cfg']['turns'] == 40 and c['cfg']['channel'] == 'blackboard' for c in g)
for x in {(lv(c), c['N']) for c in g}:
    assert sorted(c['seed'] for c in g if (lv(c), c['N']) == x) == list(range(900, 905))
# the N 8 clean baseline and request cells equal Phase 13's (same requests: cached); nothing else is
same = [c for c in g if requestKey(c) in {requestKey(x) for x in fixGrid()}]
assert sorted((lv(c), c['N'], c['seed']) for c in same) == sorted((a, 8, s) for a in SCALE_CLEAN for s in range(900, 905))
for c in same:
    ref = next(x for x in fixGrid() if requestKey(x) == requestKey(c))
    assert ref['N'] == 8 and ref['level'] == lv(c) and ref['cfg'] == c['cfg']
# the board grows with N: 2N + 2 open subtasks
from explRL.clue_runs import clueSetup
for N in SCALE_N:
    b, _ = clueSetup(LLMConfig.fromDict(next(c for c in g if c['N'] == N)['cfg']), 900)
    assert len(b.open) == 2 * N + 2
# calls: (N - liars) x 40 per cell; 39 200 in all, about 33 000 new; the pilot's two cells are cached for the main run
calls = lambda N, liar: (N - liar * (N // 4)) * 40
eg = scaleEstimate(g)
assert eg['calls'] == 5 * sum(2 * calls(N, 0) + 2 * calls(N, 1) for N in SCALE_N) == 39200
assert eg['newCalls'] == eg['calls'] - 5 * 2 * calls(8, 0) - calls(32, 0) - calls(32, 1) == 33760, eg
p = scalePilotGrid()
assert [(lv(c), c['N'], c['seed'], c['cfg']['nLiar']) for c in p] == [('baseline', 32, 900, 0), ('baseline|liar', 32, 900, 8)]
assert {requestKey(c) for c in p} <= {requestKey(c) for c in g} and all(c['study'] == 'scalePilot' for c in p)
ep = fixEstimate(p)
assert ep['calls'] == ep['newCalls'] == 32 * 40 + 24 * 40 == 2240
# the cost rule (FR-B2): full grid within $40, else N 32 on seeds 900-902, else stop and ask
cut = scaleGrid(seeds32=range(900, 903))
assert len(cut) == 4 * (5 + 5 + 3) and {c['seed'] for c in cut if c['N'] == 32} == {900, 901, 902}
nf, nc = scaleEstimate(g)['newCalls'], scaleEstimate(cut)['newCalls']
assert costRule(40.0 / nf)['verdict'] == 'run' and costRule(40.0 / nf)['cells'] == g
r = costRule(40.0 / nf * 1.01)
assert r['verdict'] == 'cut' and r['cells'] == cut and r['projected']['full'] > 40 >= r['projected']['cut']
r = costRule(40.0 / nc * 1.01)
assert r['verdict'] == 'stop' and r['cells'] is None
# quick grids: every cell kind, a few turns
assert {lv(c) for c in trustQuick()} == {lv(c) for c in t} and all(c['cfg']['turns'] == 6 for c in trustQuick())
assert len(scaleQuick()) == 12 and all(c['cfg']['turns'] == 3 for c in scaleQuick()) and all(c['cfg']['turns'] == 40 for c in g)
# the plans: version 1, not frozen yet, JSON-serialisable with a stable hash; the items of the analysis plans
for P in (TRUST_PLAN, SCALE_PLAN):
    assert P['version'] == 1 and P['frozen'] is False and P['exploratory'] is True
    assert coopHash(json.loads(json.dumps(P))) == coopHash(P)
assert set(TRUST_PLAN['analyses']) == {'lossRecovered', 'publicity', 'cost', 'difference', 'noLiarCost', 'mechanism'}
assert TRUST_PLAN['holm'] == ['recordPrivate', 'verifyPublic'] and TRUST_PLAN['arms']['world']['recordPrivate'] == dict(record='private')
assert set(SCALE_PLAN['analyses']) == {'request', 'record', 'baseline', 'forecast'}
assert SCALE_PLAN['costRule'] == dict(SCALE_PLAN['costRule'], budget=40.0, cutN=32, cutSeeds=[900, 902])
assert SCALE_PLAN['bands'] == dict(invalid=0.05, N=32) and SCALE_PLAN['holm'] == dict(request=[8, 16, 32], record=[8, 16, 32])
print(f"trustGrid {e['calls']} calls ${e['newCost']:.2f}; scaleGrid {eg['newCalls']} new calls (profile ${eg['newCost']:.2f}); "
      f"pilot {ep['calls']} calls (profile ${ep['cost']:.2f})")

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()